# 05 Predict: dry run on São Paulo, then Las Vegas

Builds the 8 features for one race **from its qualifying session and earlier races only**, refits the frozen model, and prints the predicted podium.

**Dry run (this version):** 2026 round 20, São Paulo, treated as if the race hadn't happened. The pass test checks that the features built here match the ones `03_features` saved in `races_clean.csv`.

**Saturday, for Las Vegas (round 21):**
1. After qualifying, from the repo root: `python download.py`. Qualifying files are fetched first, so `data/raw/2026_21_Q.csv` arrives early. If FastF1 doesn't have it yet, make that CSV by hand from the official results, with the columns `Abbreviation, DriverId, TeamName, Position, Q1, Q2, Q3, Year, Round` (times like `0 days 00:01:33.602000`).
2. In the settings cell: `ROUND = 21`, `DRY_RUN = False`, and type `STARTING_GRID` from the FIA's official starting grid (after penalties). In 2026, penalties move a lot of drivers.
3. Run all.

In [11]:
import glob
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

raw_path = "../data/raw/"
out_path = "../data/results/"

# ---- Settings: the only cell to change on Saturday ----
YEAR = 2026
ROUND = 21          # dry run: São Paulo. Las Vegas is round 21
DRY_RUN = False      # True = the race has already happened, so the prediction is checked against the real result

# Official starting grid in order, after penalties. Pit-lane starters go at the end.
# None = use qualifying order as the grid.
# Dry run: São Paulo's real starting grid. Penalties moved six drivers back from their qualifying spot
# (RUS 2 -> 7, LEC 4 -> 14, LIN 14 -> 18, HAM 11 -> 20, VER 16 -> 21, PER 21 -> 22)
STARTING_GRID = ['VER', 'RUS', 'NOR', 'HAD', 'LIN', 'ANT', 'LAW', 'GAS', 'COL', 'LEC', 'HAM',
                 'HUL', 'BOR', 'ALB', 'CAM', 'PER', 'PIA', 'ALO', 'BOT', 'SAI', 'STR', 'BEA' ]

# Same as 04_model: LogGrid (log of grid position) replaces raw Grid
FEATURES = ['LogGrid', 'DriverForm', 'TeamForm', 'ConstructorPoints', 'DriverShare',
            'Reliability', 'PercentageDifference', 'GapToTeammate']

## Feature functions

Copied unchanged from `03_features.ipynb`, so the prediction uses exactly the same logic the model was trained on. The pass test further down proves it.

In [12]:
def form(races, by, window=5):
    """Average finish over the previous `window` races (shifted, so race N only sees races before it). Resets every season.
    by='Abbreviation' gives driver form, by='TeamName' gives team form."""
    per_race = races.groupby(by=['Year','Round',by], as_index=False)['Position'].mean()
    per_race = per_race.sort_values(by=['Year','Round'])
    per_season = per_race.groupby(by=['Year',by])['Position']
    per_race['Form'] = per_season.transform(lambda s: s.shift().rolling(window, min_periods=1).mean())
    merged = races[['Year','Round',by]].merge(per_race[['Year','Round',by,'Form']], on=['Year','Round',by], how='left')
    merged.index = races.index
    return merged['Form']


def gap_to_teammate(df):
    """% gap to the teammate's best qualifying lap (negative = faster). No teammate: 0."""
    team = df.groupby(['Year', 'Round', 'TeamName'])['Best']
    teammate_best = (team.transform('sum') - df['Best']) / (team.transform('count') - 1)
    gap = (df['Best'] - teammate_best) * 100 / teammate_best
    return gap.fillna(0)


def points_before(races, sprints):
    """Constructor points going into each race and the driver's share of them. Resets every season.
    A sprint runs before the race on the same weekend, so its points already count."""
    sprint_points = sprints.groupby(by=['Year','Round','Abbreviation'], as_index=False)['Points'].sum().rename(columns={'Points':'SprintPoints'})
    per_race = races[['Year','Round','Abbreviation','TeamName','Points']].merge(sprint_points, on=['Year','Round','Abbreviation'], how='left')
    per_race['SprintPoints'] = per_race['SprintPoints'].fillna(0)
    per_race['WeekendPoints'] = per_race['Points'] + per_race['SprintPoints']
    per_race = per_race.sort_values(by=['Year','Round'])
    per_race['DriverPoints'] = per_race.groupby(by=['Year','TeamName','Abbreviation'])['WeekendPoints'].cumsum() - per_race['Points']
    team = per_race.groupby(by=['Year','Round','TeamName'], as_index=False)[['WeekendPoints','Points']].sum()
    team['ConstructorPoints'] = team.groupby(by=['Year','TeamName'])['WeekendPoints'].cumsum() - team['Points']
    per_race = per_race.merge(team[['Year','Round','TeamName','ConstructorPoints']], on=['Year','Round','TeamName'], how='left')
    per_race['DriverShare'] = np.where(per_race['ConstructorPoints'] > 0, per_race['DriverPoints'] / per_race['ConstructorPoints'], 0.5)
    merged = races[['Year','Round','Abbreviation']].merge(per_race[['Year','Round','Abbreviation','ConstructorPoints','DriverShare']], on=['Year','Round','Abbreviation'], how='left')
    merged.index = races.index
    return merged[['ConstructorPoints','DriverShare']]

## Load the data

- `races_clean.csv`: race history with the features already built. Only races **before** the target are used.
- Sprints straight from `data/raw/`, so a sprint on the target weekend counts. It runs before the race.
- The target race's qualifying straight from its downloaded CSV, which is the same file Saturday's download produces.

In [13]:
races = pd.read_csv(out_path + 'races_clean.csv')
races['LogGrid'] = np.log(races['Grid'])   # same as 04_model; Grid stays raw for the baselines
sprints = pd.concat([pd.read_csv(f) for f in glob.glob(raw_path + '*_S.csv')], ignore_index=True)
quali = pd.read_csv(raw_path + f'{YEAR}_{ROUND:02d}_Q.csv')

before = (races['Year'] < YEAR) | ((races['Year'] == YEAR) & (races['Round'] < ROUND))
print(f"History: {before.sum()} driver-races, up to {YEAR} round {races.loc[before & (races['Year'] == YEAR), 'Round'].max()}")
print(f"Qualifying: {quali['EventName'].iloc[0]}, {len(quali)} drivers")
quali[['Position', 'Abbreviation', 'TeamName', 'Q1', 'Q2', 'Q3']]

History: 2278 driver-races, up to 2026 round 20
Qualifying: Las Vegas Grand Prix, 22 drivers


,Position,Abbreviation,TeamName,Q1,Q2,Q3
0,1.0,ANT,Mercedes,0 days 00:01:34.684000,0 days 00:01:35.086000,0 days 00:01:33.893000
1,2.0,VER,Red Bull Racing,0 days 00:01:35.264000,0 days 00:01:34.394000,0 days 00:01:34.151000
2,3.0,RUS,Mercedes,0 days 00:01:34.664000,0 days 00:01:34.617000,0 days 00:01:34.224000
3,4.0,NOR,McLaren,0 days 00:01:35.109000,0 days 00:01:34.766000,0 days 00:01:34.244000
4,5.0,HAD,Red Bull Racing,0 days 00:01:35.413000,0 days 00:01:35.064000,0 days 00:01:34.592000
5,6.0,LIN,Racing Bulls,0 days 00:01:35.456000,0 days 00:01:34.963000,0 days 00:01:34.668000
6,7.0,PIA,McLaren,0 days 00:01:35.491000,0 days 00:01:34.763000,0 days 00:01:34.899000
7,8.0,LAW,Racing Bulls,0 days 00:01:35.489000,0 days 00:01:35.048000,0 days 00:01:35.090000
8,9.0,GAS,Alpine,0 days 00:01:35.718000,0 days 00:01:34.559000,0 days 00:01:35.105000
9,10.0,COL,Alpine,0 days 00:01:35.930000,0 days 00:01:34.982000,0 days 00:01:35.314000


## Qualifying features

The same steps as `02_build_dataset` (`Best` = fastest of Q1 to Q3) and `03_features` (no time → the session's slowest, gap to pole, gap to teammate).

In [14]:
quali[['Q1', 'Q2', 'Q3']] = quali[['Q1', 'Q2', 'Q3']].apply(lambda c: pd.to_timedelta(c).dt.total_seconds())
quali['Best'] = np.nanmin(quali[['Q1', 'Q2', 'Q3']].to_numpy(), axis=1)
quali['Best'] = quali['Best'].fillna(quali['Best'].max())

pole = quali['Best'].min()
quali['PercentageDifference'] = (quali['Best'] - pole) * 100 / pole
quali['GapToTeammate'] = gap_to_teammate(quali)

quali[['Position', 'Abbreviation', 'TeamName', 'Best', 'PercentageDifference', 'GapToTeammate']].round(3)

,Position,Abbreviation,TeamName,Best,PercentageDifference,GapToTeammate
0,1.0,ANT,Mercedes,93.893,0.000,-0.351
1,2.0,VER,Red Bull Racing,94.151,0.275,-0.466
2,3.0,RUS,Mercedes,94.224,0.353,0.353
3,4.0,NOR,McLaren,94.244,0.374,-0.548
4,5.0,HAD,Red Bull Racing,94.592,0.744,0.468
5,6.0,LIN,Racing Bulls,94.668,0.825,-0.400
6,7.0,PIA,McLaren,94.763,0.927,0.551
7,8.0,LAW,Racing Bulls,95.048,1.230,0.401
8,9.0,GAS,Alpine,94.559,0.709,-0.445
9,10.0,COL,Alpine,94.982,1.160,0.447


## Placeholder rows for the race that hasn't happened

One row per driver, with the result left blank. Appended to the history, the same feature functions then build this race's features, and `shift(1)` makes sure they only see earlier races.

`Points` is 0, not blank: `points_before` does `cumsum() - Points`, and one NaN would blank the result.

In [15]:
upcoming = quali[['Abbreviation', 'DriverId', 'TeamName', 'Year', 'Round', 'Position']].rename(columns={'Position': 'QualiPosition'})

if STARTING_GRID is None:
    upcoming['Grid'] = upcoming['QualiPosition']
else:
    assert sorted(STARTING_GRID) == sorted(upcoming['Abbreviation']), 'STARTING_GRID must list every qualifier exactly once'
    upcoming['Grid'] = upcoming['Abbreviation'].map({abbr: i + 1 for i, abbr in enumerate(STARTING_GRID)})

upcoming['Position'] = np.nan
upcoming['Points'] = 0.0
upcoming['ClassifiedFlag'] = np.nan
upcoming['Podium'] = np.nan

# Drivers who start behind their qualifying spot were penalised (everyone behind them moves up one place)
dropped = upcoming[upcoming['Grid'] > upcoming['QualiPosition']]
print(f"{len(dropped)} drivers start behind their qualifying position: {dropped['Abbreviation'].tolist()}")
upcoming.sort_values('Grid')[['Abbreviation', 'TeamName', 'QualiPosition', 'Grid']]

4 drivers start behind their qualifying position: ['ANT', 'PIA', 'SAI', 'STR']


,Abbreviation,TeamName,QualiPosition,Grid
1,VER,Red Bull Racing,2.0,1
2,RUS,Mercedes,3.0,2
3,NOR,McLaren,4.0,3
4,HAD,Red Bull Racing,5.0,4
5,LIN,Racing Bulls,6.0,5
0,ANT,Mercedes,1.0,6
7,LAW,Racing Bulls,8.0,7
8,GAS,Alpine,9.0,8
9,COL,Alpine,10.0,9
10,LEC,Ferrari,11.0,10


## Build the 8 features

History before the target, plus the placeholder rows, through the same functions and median fills as `03_features`. Round-1 medians come from the training years (before 2025) only.

In [16]:
history = races.loc[before, ['Abbreviation', 'DriverId', 'TeamName', 'Year', 'Round',
                              'Position', 'Points', 'ClassifiedFlag', 'Podium', 'Grid']]
table = pd.concat([history, upcoming.drop(columns='QualiPosition')], ignore_index=True)

table['DriverForm'] = form(table, 'Abbreviation')
table['TeamForm'] = form(table, 'TeamName')
table[['ConstructorPoints', 'DriverShare']] = points_before(table, sprints)

per_race = table[['Year', 'Round', 'Abbreviation', 'ClassifiedFlag']].sort_values(by=['Year', 'Round'])
table['Reliability'] = per_race.groupby(by=['Year', 'Abbreviation'])['ClassifiedFlag'].transform(lambda s: s.shift().expanding().mean())

train_years = table['Year'] < 2025
for col in ['DriverForm', 'TeamForm', 'Reliability']:
    table[col] = table[col].fillna(table.loc[train_years, col].median())

target = table[(table['Year'] == YEAR) & (table['Round'] == ROUND)].copy()
target = target.merge(quali[['Abbreviation', 'PercentageDifference', 'GapToTeammate']], on='Abbreviation', how='left')
target['LogGrid'] = np.log(target['Grid'])   # from the official grid, so penalties are included

blanks = target[FEATURES].isna().sum()
print('Blank features:', blanks[blanks > 0].to_dict() or 'none')
target.sort_values('Grid')[['Abbreviation', 'TeamName', 'Grid'] + FEATURES].round(3)

Blank features: none


,Abbreviation,TeamName,Grid,LogGrid,DriverForm,TeamForm,ConstructorPoints,DriverShare,Reliability,PercentageDifference,GapToTeammate
1,VER,Red Bull Racing,1.0,0.000,1.6,4.1,415.0,0.672,0.850,0.275,-0.466
2,RUS,Mercedes,2.0,0.693,9.4,6.1,652.0,0.402,0.900,0.353,0.353
3,NOR,McLaren,3.0,1.099,4.8,6.1,398.0,0.618,0.800,0.374,-0.548
4,HAD,Red Bull Racing,4.0,1.386,6.6,4.1,415.0,0.294,0.882,0.744,0.468
5,LIN,Racing Bulls,5.0,1.609,13.2,10.3,114.0,0.333,0.900,0.825,-0.400
0,ANT,Mercedes,6.0,1.792,2.8,6.1,652.0,0.598,1.000,0.000,-0.351
7,LAW,Racing Bulls,7.0,1.946,7.4,10.3,114.0,0.658,0.950,1.230,0.401
8,GAS,Alpine,8.0,2.079,13.6,15.1,69.0,0.609,0.900,0.709,-0.445
9,COL,Alpine,9.0,2.197,16.6,15.1,69.0,0.391,0.850,1.160,0.447
10,LEC,Ferrari,10.0,2.303,4.0,4.4,510.0,0.488,0.900,1.222,-0.224


## Pass test (dry run only)

`races_clean.csv` already holds São Paulo's features, built by `03_features` and checked for leakage on Wednesday. If the features built here from qualifying plus earlier races match them, the prediction step builds features exactly as the model was trained.

In [17]:
if DRY_RUN:
    stored = races[(races['Year'] == YEAR) & (races['Round'] == ROUND)][['Abbreviation'] + FEATURES]
    check = target[['Abbreviation'] + FEATURES].merge(stored, on='Abbreviation', suffixes=('_built', '_stored'))
    print(f"Compared {len(check)} drivers")
    all_ok = True
    for f in FEATURES:
        ok = np.allclose(check[f + '_built'], check[f + '_stored'])
        all_ok &= ok
        print(f"  {f:<22} {'match' if ok else 'DIFFERENT'}")
    print('PASS' if all_ok else 'FAIL: look at the DIFFERENT features before trusting the prediction')
else:
    print('Not a dry run: no stored features to compare against')

Not a dry run: no stored features to compare against


## Refit the frozen model

Same model and settings as frozen in the README (`LogisticRegression()`, default `C=1.0`, `StandardScaler`), refitted on every race from 2022 up to the one before the target.

In [18]:
train = races.loc[before]
scaler = StandardScaler().fit(train[FEATURES])
model = LogisticRegression().fit(scaler.transform(train[FEATURES]), train['Podium'])

print(f"Trained on {len(train)} driver-races ({train['Year'].min()} to {YEAR} round {train.loc[train['Year'] == YEAR, 'Round'].max()})")
pd.Series(model.coef_[0], index=FEATURES).round(2).to_frame('coefficient')

Trained on 2278 driver-races (2022 to 2026 round 20)


,coefficient
LogGrid,-1.47
DriverForm,-0.16
TeamForm,-0.88
ConstructorPoints,0.08
DriverShare,0.26
Reliability,-0.11
PercentageDifference,0.03
GapToTeammate,-0.06


## Prediction

Each driver's podium probability, rescaled so the race adds up to its 3 podium places (capped at 100%), then ordered from most to least likely with `np.argsort(-p)`.

In [19]:
p = model.predict_proba(scaler.transform(target[FEATURES]))[:, 1]
target['p_raw'] = p
target['p'] = np.minimum(3 * p / p.sum(), 1)

order = np.argsort(-target['p'].to_numpy())
ranked = target.iloc[order].reset_index(drop=True)

print(f"Predicted podium, {YEAR} round {ROUND} ({quali['EventName'].iloc[0]}):")
for i, row in ranked.head(3).iterrows():
    print(f"  P{i + 1}  {row['Abbreviation']}  ({row['TeamName']}, grid {int(row['Grid'])})  {row['p']:.0%}")


ranked[['Abbreviation', 'TeamName', 'Grid', 'p_raw', 'p'] + FEATURES[1:]].round(3)

Predicted podium, 2026 round 21 (Las Vegas Grand Prix):
  P1  VER  (Red Bull Racing, grid 1)  85%
  P2  RUS  (Mercedes, grid 2)  58%
  P3  NOR  (McLaren, grid 3)  50%


,Abbreviation,TeamName,Grid,p_raw,p,DriverForm,TeamForm,ConstructorPoints,DriverShare,Reliability,PercentageDifference,GapToTeammate
0,VER,Red Bull Racing,1.0,0.944,0.848,1.6,4.1,415.0,0.672,0.850,0.275,-0.466
1,RUS,Mercedes,2.0,0.647,0.581,9.4,6.1,652.0,0.402,0.900,0.353,0.353
2,NOR,McLaren,3.0,0.555,0.499,4.8,6.1,398.0,0.618,0.800,0.374,-0.548
3,HAD,Red Bull Racing,4.0,0.415,0.372,6.6,4.1,415.0,0.294,0.882,0.744,0.468
4,ANT,Mercedes,6.0,0.260,0.233,2.8,6.1,652.0,0.598,1.000,0.000,-0.351
5,LEC,Ferrari,10.0,0.147,0.132,4.0,4.4,510.0,0.488,0.900,1.222,-0.224
6,HAM,Ferrari,11.0,0.121,0.109,4.8,4.4,510.0,0.512,0.950,1.448,0.224
7,LIN,Racing Bulls,5.0,0.080,0.072,13.2,10.3,114.0,0.333,0.900,0.825,-0.400
8,LAW,Racing Bulls,7.0,0.071,0.064,7.4,10.3,114.0,0.658,0.950,1.230,0.401
9,PIA,McLaren,17.0,0.033,0.030,7.4,6.1,398.0,0.382,0.850,0.927,0.551


## Score against the real result (dry run only)

In [20]:
if DRY_RUN:
    actual = races[(races['Year'] == YEAR) & (races['Round'] == ROUND)][['Abbreviation', 'Position', 'Podium', 'Status']]
    result = ranked[['Abbreviation', 'Grid', 'p']].merge(actual, on='Abbreviation', how='left')
    result.insert(0, 'PredRank', np.arange(1, len(result) + 1))
    predicted = result.head(3)
    real_podium = actual.sort_values('Position').head(3)['Abbreviation'].tolist()
    print(f"Predicted: {predicted['Abbreviation'].tolist()}")
    print(f"Actual:    {real_podium}")
    print(f"Model: winner right {predicted.iloc[0]['Position'] == 1}, podium hits {int(predicted['Podium'].sum())} / 3")

    grid_top3 = result[result['Grid'] <= 3]
    pole_won = (result.loc[result['Grid'] == 1, 'Position'] == 1).any()
    print(f"Grid:  pole won {pole_won}, grid top 3 podium hits {int(grid_top3['Podium'].sum())} / 3")
    display(result.head(10).round(3))
    actual = actual.merge(right = result[['Abbreviation','p']],on='Abbreviation', how='left')
    display(actual.head(10).round(3))
else:
    print('Not a dry run: the result comes on Sunday')

Not a dry run: the result comes on Sunday
